In [ ]:
import bs4
import lxml
import requests
from IPython.display import clear_output
from tqdm import tqdm

from paths import paths, prepare_preprocessing

In [ ]:
prepare_preprocessing()

In [ ]:
OUT_DIR = paths.raw.health.patients
BASE_URL = 'https://digital.nhs.uk/data-and-information/publications/statistical/patients-registered-at-a-gp-practice/'

YEARS = [str(year) for year in range(2021, 2027)]

FILE_PREFIXES = ['gp-reg-pat-prac-lsoa-2021-male-female', 'gp-reg-pat-prac-lsoa-male-female']

In [ ]:
# Fetch GP registration data for each year
def download_gp_data(link, out_filename):
    resp = requests.get(link, stream = True)
    resp.raise_for_status()

    total = int(resp.headers.get('Content-Length', 0))
    file_path = OUT_DIR / out_filename
    OUT_DIR.mkdir(parents = True, exist_ok = True)

    with open(file_path, 'wb') as f:
        with tqdm(total = total, unit = 'B', unit_scale = True) as pbar:
            for chunk in resp.iter_content(chunk_size = 1024 * 1024):
                if not chunk:
                    continue

                f.write(chunk)
                pbar.update(len(chunk))

links = []
for year in YEARS[::-1]:
    res = requests.get(f'{BASE_URL}/april-{year}')
    soup = bs4.BeautifulSoup(res.text, 'lxml')

    link_element = soup.find('a', href = lambda href: any(prefix in href for prefix in FILE_PREFIXES))

    if not link_element:
        continue

    links.append((link_element['href'], f'{year}.zip'))

for link, filename in links:
    if (OUT_DIR / filename).exists():
        continue

    print(f'Downloading {filename}...')
    download_gp_data(link, filename)
    clear_output(wait = True)